# 09 — Final test evaluation
The only notebook that reads the test set. Each checkpoint is scored once and the result is stored.
Can run before notebook 10 (seed 42 only); run it again after notebook 10 to add seeds 43/44 — the seed-42 results are reused, not recomputed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git -C /content/realwaste-classification pull -q || git clone -q https://github.com/KNV-DSEB/realwaste-classification.git /content/realwaste-classification
import sys
sys.path.insert(0, "/content/realwaste-classification")
for m in [m for m in sys.modules if m.split(".")[0] == "src"]:
    del sys.modules[m]  # re-import the code just pulled

In [ ]:
import pandas as pd
import torch
from IPython.display import Image, display
from src.dataset import load_frozen_split
from src.final_eval import comparison_table, evaluate_experiment, per_class_f1_table
from src.utils import get_paths, load_config

cfg = load_config()
split_df = load_frozen_split(cfg)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
res_dir = get_paths(cfg).result_dir
(res_dir / "final").mkdir(exist_ok=True)

In [ ]:
results = {exp: evaluate_experiment(cfg, exp, split_df, device, amp=device.type == "cuda")
           for exp in ["E1", "E2", "E3", "E4"]}

table = comparison_table(results)
per_class = per_class_f1_table(cfg, list(results))
table.to_csv(res_dir / "final" / "model_comparison.csv", index=False)
per_class.to_csv(res_dir / "final" / "per_class_f1.csv")
print(table.to_string(index=False))
print(per_class.round(3).to_string())

In [ ]:
for exp in results:
    display(Image(filename=str(res_dir / "experiments" / exp / f"{exp}_confusion_matrix.png"), width=500))

In [ ]:
# error analysis uses the model with the best VALIDATION macro-F1, never picked by test results
best = table.loc[table["best_val_macro_f1"].idxmax(), "experiment"]
errors = pd.read_csv(res_dir / "experiments" / best / f"{best}_errors.csv")
print(best, len(errors), "misclassified test images")
print(errors.groupby(["true_class", "predicted_class"]).size().sort_values(ascending=False).head(10))